# Flight Delay Analysis
**Dataset:** 2015 US domestic flights (~5.8M rows)  
**Goal:** Predict arrival/departure delays

## 1. Setup & Imports

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('..')

## 2. Data Loading

In [ ]:
# Explicit dtypes cut memory usage roughly in half
FLIGHT_DTYPES = {
   # 'YEAR': 'int16', # alles 2015 -> raus
    
    'MONTH': 'int8', 
    'DAY': 'int8', 
    'DAY_OF_WEEK': 'int8',

    'AIRLINE':'string',

    'FLIGHT_NUMBER': 'int32', 
    
    #'TAIL_NUMBER' viele missing 

    'ORIGIN_AIRPORT':'string',
    'DESTINATION_AIRPORT':'string',

    'SCHEDULED_DEPARTURE': 'string',  
    'DEPARTURE_TIME': 'string', #1% missing -> raus
    'DEPARTURE_DELAY': 'float32', #1% missing = DEPARTURE_TIME


    #'TAXI_OUT': 'float32', # 2% viele missing -> raus
    #'WHEELS_OFF': 'string', # 2% viele missing -> raus = TAX_OFF

    'SCHEDULED_TIME': 'float32', # 6 mising-> raus
    'ELAPSED_TIME': 'float32',# 2% viele missing -> raus
    #'AIR_TIME': 'float32',# 2% viele missing -> raus =ELAPSED
    'DISTANCE': 'int32',
    
    
    #'WHEELS_ON': 'string',  # # 2% viele missing -> raus
    #'TAXI_ON': 'float32', # 2% viele missing -> raus
    'SCHEDULED_ARRIVAL' : 'string', 
    'ARRIVAL_TIME': 'string',
    'ARRIVAL_DELAY': 'float32', 

    
    #'DIVERTED': 'bool', # raus -> es gibt nur 0
    #'CANCELLED': 'bool',  # 99.99% sind 0 -> raus

    #'AIRSYSTEM_DELAY'  _> raus
    #'SECURITY_DELAY'  _> raus
    #'AIRLINE_DELAY'  _> raus
    #'LATEAIRCRAFT_DELAY'  _> raus
    #'WEATHER_DELAY'  _> raus

}

print('Loading flights.csv (~592 MB) ...')
flights = pd.read_csv(DATA_DIR / 'flights.csv', dtype=FLIGHT_DTYPES)
airlines = pd.read_csv(DATA_DIR / 'airlines.csv')
airports = pd.read_csv(DATA_DIR / 'airports.csv')

# Keep only columns defined in FLIGHT_DTYPES
flights = flights[list(FLIGHT_DTYPES.keys())]
before = print(len(flights))

# Drop rows with any missing value


print(len(flights))

print(f'flights: {flights.shape[0]:,} rows × {flights.shape[1]} cols')
print(f'airlines: {airlines.shape[0]} | airports: {airports.shape[0]}')

Loading flights.csv (~592 MB) ...
5819079
5819079
flights: 5,819,079 rows × 16 cols
airlines: 14 | airports: 322


## 2a. Datenbereinigung

In [35]:
valid_airlines = set(airlines['IATA_CODE'])
before = len(flights)
invalid_airlines = flights.loc[~flights['AIRLINE'].isin(valid_airlines), 'AIRLINE'].value_counts()
flights = flights[flights['AIRLINE'].isin(valid_airlines)]
print(f'Airline-Filter: {before - len(flights):,} Zeilen entfernt → {len(flights):,} verbleibend')
print('\nEntfernte Airline IATA-Codes:')
print(invalid_airlines.to_string() if len(invalid_airlines) else '  keine')

Airline-Filter: 0 Zeilen entfernt → 5,714,008 verbleibend

Entfernte Airline IATA-Codes:
  keine


In [36]:
valid_airports = set(airports['IATA_CODE'])
before = len(flights)
invalid_origins = flights.loc[~flights['ORIGIN_AIRPORT'].isin(valid_airports), 'ORIGIN_AIRPORT'].value_counts()
invalid_dests = flights.loc[~flights['DESTINATION_AIRPORT'].isin(valid_airports), 'DESTINATION_AIRPORT'].value_counts()
flights = flights[
    flights['ORIGIN_AIRPORT'].isin(valid_airports) &
    flights['DESTINATION_AIRPORT'].isin(valid_airports)
]
print(f'Airport-Filter: {before - len(flights):,} Zeilen entfernt → {len(flights):,} verbleibend')
print('\nEntfernte ORIGIN_AIRPORT-Codes:')
print(invalid_origins.to_string() if len(invalid_origins) else '  keine')
print('\nEntfernte DESTINATION_AIRPORT-Codes:')
print(invalid_dests.to_string() if len(invalid_dests) else '  keine')

Airport-Filter: 482,878 Zeilen entfernt → 5,231,130 verbleibend

Entfernte ORIGIN_AIRPORT-Codes:
ORIGIN_AIRPORT
10397    32509
13930    27566
11298    20586
11292    18077
12892    17628
14771    14094
12266    13091
14107    12884
12889    12649
13487    10552
14747    10361
10721    10091
11433     9882
11057     9537
11618     9508
13204     9006
14869     8738
12953     8447
12478     8255
10821     7991
13232     7598
11278     6759
14679     6184
14100     6065
13303     5929
11259     5873
11697     5753
15304     5125
12191     4681
14057     4539
10693     4450
15016     4295
13796     3994
10423     3871
12173     3761
13495     3646
14893     3598
13198     3536
14908     3524
14831     3457
11042     3202
14492     2984
12264     2979
13342     2631
14683     2594
12339     2361
11066     2178
14122     2122
11193     1903
14843     1858
13830     1814
10140     1805
14635     1756
10800     1753
13891     1734
14027     1734
10529     1705
12451     1701
14524     1628
138

In [37]:
out_path = DATA_DIR / 'flights_cut.csv'
flights.to_csv(out_path, index=False)
print(f'Saved {flights.shape[0]:,} rows × {flights.shape[1]} cols → {out_path}')

Saved 5,231,130 rows × 16 cols → ../flights_cut.csv


## 2b. Train / Validation Split

In [12]:
from sklearn.model_selection import train_test_split

# 60% train / 20% test / 20% val
train_test, val = train_test_split(flights, test_size=0.2, random_state=42)
train, test = train_test_split(train_test, test_size=0.25, random_state=42)

train.to_csv(DATA_DIR / 'flights_train.csv', index=False)
test.to_csv(DATA_DIR / 'flights_test.csv', index=False)
val.to_csv(DATA_DIR / 'flights_val.csv', index=False)

total = len(flights)
print(f'Train: {len(train):,} rows ({len(train)/total:.0%})')
print(f'Test:  {len(test):,} rows ({len(test)/total:.0%})')
print(f'Val:   {len(val):,} rows ({len(val)/total:.0%})')

Train: 3,428,404 rows (60%)
Test:  1,142,802 rows (20%)
Val:   1,142,802 rows (20%)


In [9]:
train.head(10)

,MONTH,DAY,DAY_OF_WEEK,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_TIME,ELAPSED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY
3995414,9,5,6,3130,1520,1514,-6.0,55.0,42.0,134,1615,1556,-19.0
4983761,11,8,7,5044,0700,0655,-5.0,125.0,132.0,618,0905,0907,2.0
2520096,6,9,2,3244,0625,0617,-8.0,72.0,69.0,173,0737,0726,-11.0
3380754,7,30,4,647,1015,1008,-7.0,76.0,75.0,325,1131,1123,-8.0
2537752,6,10,3,2090,0700,0655,-5.0,193.0,190.0,1426,1213,1205,-8.0
5068810,11,13,5,5030,1025,1015,-10.0,138.0,156.0,780,1243,1251,8.0
1251453,3,22,7,351,1741,1902,81.0,130.0,109.0,599,1851,1951,60.0
336642,1,23,5,1456,0600,0556,-4.0,93.0,95.0,356,0733,0731,-2.0
4678980,10,19,1,2814,1450,1454,4.0,199.0,173.0,1089,1809,1747,-22.0
3810376,8,25,2,5661,0730,0728,-2.0,90.0,128.0,429,0900,0936,36.0


## 3. Exploratory Data Analysis

In [ ]:
# Null overview
null_pct = flights.isnull().mean().mul(100).sort_values(ascending=False)
print('Columns with missing values:')
print(null_pct[null_pct > 0].to_string())

In [ ]:
# Arrival delay distribution (non-cancelled flights)
on_time = flights[flights['CANCELLED'] == 0].copy()
print(f'Non-cancelled flights: {len(on_time):,}')
print(f'Delayed >15 min: {(on_time["ARRIVAL_DELAY"] > 15).mean():.1%}')
print(f'Cancelled: {flights["CANCELLED"].mean():.1%}')

on_time['ARRIVAL_DELAY'].clip(-30, 120).hist(bins=60, edgecolor='none')
plt.axvline(15, color='red', linestyle='--', label='15 min threshold')
plt.xlabel('Arrival Delay (min)')
plt.ylabel('Flights')
plt.title('Arrival Delay Distribution')
plt.legend()
plt.tight_layout()

In [ ]:
# Average delay by month
monthly = on_time.groupby('MONTH')['ARRIVAL_DELAY'].mean()
monthly.plot(kind='bar', color='steelblue')
plt.xlabel('Month')
plt.ylabel('Mean Arrival Delay (min)')
plt.title('Average Delay by Month')
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
# Delay by airline
airline_delay = (
    on_time.merge(airlines, left_on='AIRLINE', right_on='IATA_CODE')
    .groupby('AIRLINE_y')['ARRIVAL_DELAY']
    .mean()
    .sort_values()
)
airline_delay.plot(kind='barh', color='steelblue')
plt.xlabel('Mean Arrival Delay (min)')
plt.title('Average Delay by Airline')
plt.tight_layout()

## 4. Feature Engineering

In [ ]:
df = on_time.dropna(subset=['ARRIVAL_DELAY']).copy()

# Cyclical time encoding
df['HOUR'] = df['SCHEDULED_DEPARTURE'] // 100
df['HOUR_SIN'] = np.sin(2 * np.pi * df['HOUR'] / 24)
df['HOUR_COS'] = np.cos(2 * np.pi * df['HOUR'] / 24)
df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)
df['DOW_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)
df['DOW_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)

# Encode categoricals
for col in ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']:
    df[col] = df[col].astype('category').cat.codes

# Binary target
df['LATE'] = (df['ARRIVAL_DELAY'] > 15).astype(int)

# Feature columns (no leakage)
LEAKAGE = [
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'DEPARTURE_TIME',
    'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'TAXI_OUT',
    'ELAPSED_TIME', 'AIR_TIME', 'ARRIVAL_TIME', 'DEPARTURE_DELAY',
    'TAIL_NUMBER', 'CANCELLATION_REASON',
]
FEATURES = [c for c in df.columns if c not in LEAKAGE + ['ARRIVAL_DELAY', 'LATE', 'CANCELLED', 'DIVERTED']]
print('Features:', FEATURES)

## 5. Modelling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

X = df[FEATURES].fillna(0)
y = df['LATE']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df['MONTH']
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print('Baseline:', classification_report(y_test, baseline.predict(X_test), digits=3))

In [ ]:
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test), digits=3))

## 6. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURES).nlargest(15)
importance.sort_values().plot(kind='barh', color='steelblue')
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.tight_layout()